# 03 · LLM steps and evaluation

Resume profiles, an LLM judge for evaluation, LLM ranking with explanations, and the data export for the app. Uses gpt-oss models via Groq's free tier. Every call is cached on disk, so re-running only spends tokens on prompts that haven't been answered before.

Needs `GROQ_API_KEY` in a `.env` file in this folder.

| Version (42 judged resumes) | Label P@10 | Judge P@10 (grade ≥ 1) | Judge P@10 (grade = 2) |
|---|---|---|---|
| Raw + profile | 0.755 | 0.814 | 0.452 |
| + chunked postings | 0.750 | 0.790 | 0.414 |
| + cross-encoder reranker | 0.752 | 0.817 | 0.412 |
| + seniority penalty | 0.757 | 0.833 | 0.452 |
| + boilerplate removal | 0.755 | 0.836 | 0.443 |
| **+ LLM ranking of the top 20** | **0.745** | **0.881** | **0.500** |

## Setup: Groq client and cached LLM wrapper

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found - check the .env file name and location"

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))
for m in sorted(client.models.list().data, key=lambda m: m.id):
    print(m.id)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo


In [ ]:
import hashlib
import json
import time
from openai import RateLimitError

BASE_DIR = r"C:\Users\LENOVO\Documents\Datasets"
OUT_DIR = os.path.join(BASE_DIR, "processed")
CACHE_DIR = os.path.join(OUT_DIR, "llm_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

BACKENDS = {
    "groq":   {"base_url": "https://api.groq.com/openai/v1", "api_key": os.getenv("GROQ_API_KEY")},
    "ollama": {"base_url": "http://localhost:11434/v1", "api_key": "ollama"},
}
BACKEND = "groq"
_client = OpenAI(**BACKENDS[BACKEND], max_retries=0)

def llm(prompt, model="openai/gpt-oss-20b", system=None, reasoning_effort="low", max_retries=8):
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": prompt}]

    # Cache: same backend + model + prompt -> reuse saved answer
    key = hashlib.sha256(json.dumps([BACKEND, model, messages, reasoning_effort]).encode()).hexdigest()
    path = os.path.join(CACHE_DIR, key + ".json")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)["content"]

    kwargs = {"reasoning_effort": reasoning_effort} if reasoning_effort else {}
    for attempt in range(max_retries):
        try:
            resp = _client.chat.completions.create(model=model, messages=messages, **kwargs)
            break
        except RateLimitError as e:
            wait = float(e.response.headers.get("retry-after", 10))
            if wait > 120:
                raise RuntimeError(f"Rate limit needs a {wait/60:.0f} min wait - daily limit likely hit") from e
            print(f"Rate limited, waiting {wait:.0f}s (attempt {attempt + 1})")
            time.sleep(wait + 1)
    else:
        raise RuntimeError("Gave up after repeated rate limits")

    content = resp.choices[0].message.content
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"model": model, "messages": messages, "content": content}, f, ensure_ascii=False)
    return content

## Resume profiles

`gpt-oss-20b` condenses each of the 210 evaluation resumes into a title, seniority, skills, domains and a short summary. Retrieval with these profiles is in notebook 02.

In [ ]:
import json
import pandas as pd

eval_clean = pd.read_pickle(os.path.join(OUT_DIR, "resumes_eval_clean.pkl"))

SYSTEM = "You extract structured profiles from resumes for job matching. Respond with JSON only, no other text."

PROMPT = """Resume:
{resume}

Return a JSON object with these keys:
"title": the candidate's most recent or target job title
"seniority": one of "entry", "mid", "senior", "executive"
"skills": list of up to 15 concrete skills, tools, or certifications
"domains": list of up to 3 industries or domains they have worked in
"summary": 2-3 sentences on what kinds of roles this person is suited for
Use only information stated in the resume."""

def parse_json(text):
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

def summarize_resume(text, max_words=1200):
    resume = " ".join(text.split()[:max_words])   # cap input to save tokens
    return parse_json(llm(PROMPT.format(resume=resume), system=SYSTEM))

In [ ]:
import unicodedata
from tqdm.auto import tqdm

PROMPT = PROMPT.replace(
    "Use only information stated in the resume.",
    "Use only information stated in the resume. Use gender-neutral language."
)

def clean_text(s):
    s = unicodedata.normalize("NFKC", s)
    return s.replace("\u2011", "-").replace("\u2010", "-")

def clean_profile(p):
    return {k: [clean_text(x) for x in v] if isinstance(v, list) else clean_text(str(v))
            for k, v in p.items()}

profiles, failed = [], []
for _, row in tqdm(eval_clean.iterrows(), total=len(eval_clean)):
    try:
        profiles.append(clean_profile(summarize_resume(row["text"])))
    except (json.JSONDecodeError, KeyError) as e:
        profiles.append(None)
        failed.append(row["ID"])

eval_clean = eval_clean.copy()
eval_clean["profile"] = profiles
print(f"Done. Failed to parse: {len(failed)}")

eval_clean.to_pickle(os.path.join(OUT_DIR, "resumes_eval_clean_profiles.pkl"))
print("Saved.")

Done. Failed to parse: 0
Saved.


## LLM judge

Category labels can't tell a good match from a merely same-category one, so `gpt-oss-120b` grades each retrieved job 0 (no match), 1 (partial) or 2 (good match) for 42 resumes (3 per category). The judge never sees the labels. Of the label "misses", 69% were judged at least a partial match, and 16% of label "hits" were judged not a match.

In [ ]:
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

corpus = pd.read_pickle(os.path.join(OUT_DIR, "jobs_corpus.pkl"))
judge_df = pd.read_pickle(os.path.join(OUT_DIR, "judge_candidates.pkl"))

JUDGE_MODEL = "openai/gpt-oss-120b"
JUDGE_SYSTEM = "You are an experienced recruiter evaluating candidate-job fit. Respond with JSON only, no other text."
JUDGE_PROMPT = """Candidate profile:
{profile}

Job postings:
{jobs}

Grade how well the candidate fits each job:
2 = good match: their experience and skills fit the role at roughly the right seniority
1 = partial match: related field or adjacent role, but with clear gaps in domain, seniority, or key skills
0 = not a match

Return a JSON object mapping each job ID to its grade, e.g. {{"J1": 2, "J2": 0}}. Grade every job."""

def profile_to_text(p):
    return (f"{p['title']}. Seniority: {p['seniority']}. {p['summary']} "
            f"Skills: {', '.join(p['skills'])}. Domains: {', '.join(p['domains'])}.")

def job_snippet(j, max_words=120):
    row = corpus.iloc[j]
    desc = " ".join(str(row["description"]).split()[:max_words])
    return f"{row['title']}: {desc}"

def parse_json(text):
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

def to_grade(g):
    try:
        g = int(g)
        return g if g in (0, 1, 2) else np.nan
    except (TypeError, ValueError):
        return np.nan

records, failed = [], []
for _, r in tqdm(judge_df.iterrows(), total=len(judge_df)):
    cands = r["candidates"]
    jobs_text = "\n\n".join(f"[J{n + 1}] {job_snippet(j)}" for n, j in enumerate(cands))
    prompt = JUDGE_PROMPT.format(profile=profile_to_text(r["profile"]), jobs=jobs_text)
    try:
        grades = parse_json(llm(prompt, model=JUDGE_MODEL, system=JUDGE_SYSTEM))
    except json.JSONDecodeError:
        failed.append(r["ID"])
        continue
    for n, j in enumerate(cands):
        records.append({"ID": r["ID"], "job_idx": j, "grade": to_grade(grades.get(f"J{n + 1}"))})

grades_df = pd.DataFrame(records)
grades_df.to_pickle(os.path.join(OUT_DIR, "judge_grades.pkl"))

print(f"Resumes failed to parse: {len(failed)}")
print(f"Pairs graded: {grades_df['grade'].notna().sum()} / {judge_df['candidates'].str.len().sum()}")
print("\nGrade distribution:")
print(grades_df["grade"].value_counts(dropna=False).sort_index().to_string())

Resumes failed to parse: 0
Pairs graded: 639 / 639

Grade distribution:
grade
0    127
1    245
2    267


In [ ]:
g = grades_df.set_index(["ID", "job_idx"])["grade"]
corpus_labels = corpus["labels"].apply(set).tolist()

rows = []
for _, r in judge_df.iterrows():
    for system in ["top_whole", "top_chunked"]:
        jobs = r[system]
        gr = np.array([g[(r["ID"], j)] for j in jobs])
        lab = np.array([r["Category"] in corpus_labels[j] for j in jobs])
        rows.append({"system": system, "ID": r["ID"], "Category": r["Category"],
                     "label P@10": lab.mean(),
                     "judge P@10 (>=1)": (gr >= 1).mean(),
                     "judge P@10 (=2)": (gr == 2).mean()})
res = pd.DataFrame(rows)

print("=== Both systems, same 42 resumes")
print(res.groupby("system")[["label P@10", "judge P@10 (>=1)", "judge P@10 (=2)"]].mean().round(3).to_string())

pairs = grades_df.merge(judge_df[["ID", "Category"]], on="ID")
pairs["label_match"] = [c in corpus_labels[j] for c, j in zip(pairs["Category"], pairs["job_idx"])]
print("\n=== Category label vs judge grade (all 639 pairs)")
print(pd.crosstab(pairs["label_match"], pairs["grade"], margins=True).to_string())

=== Both systems, same 42 resumes
             label P@10  judge P@10 (>=1)  judge P@10 (=2)
system                                                    
top_chunked       0.750             0.790            0.414
top_whole         0.755             0.814            0.452

=== Category label vs judge grade (all 639 pairs)
grade          0    1    2  All
label_match                    
False         51   60   52  163
True          76  185  215  476
All          127  245  267  639


### Spot-check

The sharpest judge-vs-label disagreements, read by hand: the judge was right in about 6 of 8, and leans strict on seniority. That strictness applies equally to every version, so comparisons stay fair.

In [ ]:
def show(rows, title):
    print(f"\n{'#' * 80}\n{title}")
    for _, p in rows.iterrows():
        prof = judge_df.set_index("ID").loc[p["ID"], "profile"]
        job = corpus.iloc[p["job_idx"]]
        print(f"\n--- [{p['Category']}] {prof['title']} ({prof['seniority']})")
        print(f"    Skills: {', '.join(prof['skills'][:8])}")
        print(f"    JOB: {job['title']}  {job['labels']}")
        print(f"    {' '.join(str(job['description']).split()[:60])}...")

show(pairs[(pairs["label_match"]) & (pairs["grade"] == 0)].sample(4, random_state=0),
     "Label says MATCH, judge says 0")
show(pairs[(~pairs["label_match"]) & (pairs["grade"] == 2)].sample(4, random_state=0),
     "Label says NO MATCH, judge says 2")

################################################################################
Label says MATCH, judge says 0

--- [ENGINEERING] Engineering Intern (mid)
    Skills: C++, Python, MATLAB, Git, Bash, R, SQL, Scikit-learn
    JOB: Mechanical Engineer Intern - Continuation  ['DESIGNER', 'ENGINEERING']
    MAJOR FUNCTION Under direction of an engineer, the intern applies basic principles & techniques of mechanical engineering & analysis in areas such as: designing components in CAD, testing in the R&D lab, failure analysis, technical tasks in support of other engineers, & focused technical design improvement projects. Essential Duties And Responsibilities Usage of CAD software for modeling & drawing creationAssist...

--- [ENGINEERING] Engineering Intern (mid)
    Skills: C++, Python, MATLAB, Git, Bash, R, SQL, Scikit-learn
    JOB: Senior Industrial Engineer  ['ENGINEERING']
    Responsibilities: Lead manufacturing-related projectsLead operations improvement - ergonomic & human factor, o

## Reranker vs plain embeddings

In [ ]:
rerank_top10 = pd.read_pickle(os.path.join(OUT_DIR, "rerank_top10.pkl"))
judge_df["top_rerank"] = judge_df["ID"].map(rerank_top10)

graded = set(zip(grades_df["ID"], grades_df["job_idx"]))
todo = {r["ID"]: [j for j in r["top_rerank"] if (r["ID"], j) not in graded]
        for _, r in judge_df.iterrows()}
print(f"New pairs to judge: {sum(len(v) for v in todo.values())} "
      f"(already graded: {sum(10 - len(v) for v in todo.values())})")

new_records = []
for _, r in tqdm(judge_df.iterrows(), total=len(judge_df)):
    new_jobs = todo[r["ID"]]
    if not new_jobs:
        continue
    jobs_text = "\n\n".join(f"[J{n + 1}] {job_snippet(j)}" for n, j in enumerate(new_jobs))
    prompt = JUDGE_PROMPT.format(profile=profile_to_text(r["profile"]), jobs=jobs_text)
    grades = parse_json(llm(prompt, model=JUDGE_MODEL, system=JUDGE_SYSTEM))
    for n, j in enumerate(new_jobs):
        new_records.append({"ID": r["ID"], "job_idx": j, "grade": to_grade(grades.get(f"J{n + 1}"))})

grades_df = pd.concat([grades_df, pd.DataFrame(new_records)], ignore_index=True)
grades_df.to_pickle(os.path.join(OUT_DIR, "judge_grades.pkl"))
judge_df.to_pickle(os.path.join(OUT_DIR, "judge_candidates.pkl"))

# Score all three systems on the same 42 resumes
g = grades_df.set_index(["ID", "job_idx"])["grade"]
rows = []
for _, r in judge_df.iterrows():
    for system in ["top_whole", "top_chunked", "top_rerank"]:
        gr = np.array([g[(r["ID"], j)] for j in r[system]])
        lab = np.array([r["Category"] in corpus_labels[j] for j in r[system]])
        rows.append({"system": system, "label P@10": lab.mean(),
                     "judge P@10 (>=1)": (gr >= 1).mean(),
                     "judge P@10 (=2)": (gr == 2).mean()})

print("\n=== All systems, same 42 resumes")
print(pd.DataFrame(rows).groupby("system").mean().round(3).to_string())

New pairs to judge: 247 (already graded: 173)

=== All systems, same 42 resumes
             label P@10  judge P@10 (>=1)  judge P@10 (=2)
system                                                    
top_chunked       0.750             0.790            0.414
top_rerank        0.752             0.817            0.412
top_whole         0.755             0.814            0.452


In [ ]:
per_resume = pd.DataFrame(rows).assign(ID=np.repeat(judge_df["ID"].values, 3))
strict = per_resume.pivot_table(index="ID", columns="system", values="judge P@10 (=2)")

for other in ["top_chunked", "top_rerank"]:
    diff = strict[other] - strict["top_whole"]
    print(f"{other} vs top_whole: better on {(diff > 0).sum()}, "
          f"worse on {(diff < 0).sum()}, tied on {(diff == 0).sum()} resumes "
          f"(mean diff {diff.mean():+.3f})")

top_chunked vs top_whole: better on 14, worse on 17, tied on 11 resumes (mean diff -0.038)
top_rerank vs top_whole: better on 9, worse on 18, tied on 15 resumes (mean diff -0.040)


## Seniority penalty and boilerplate cleaning

`judge_system` grades only jobs not graded before. Each change is also checked directly: do the jobs it added grade higher than the ones it removed?

In [ ]:
def judge_system(name, top10_file):
    """Judge only ungraded jobs in a system's top-10 lists; returns updated grades_df."""
    tops = pd.read_pickle(os.path.join(OUT_DIR, top10_file))
    judge_df[name] = judge_df["ID"].map(tops)
    graded = set(zip(grades_df["ID"], grades_df["job_idx"]))
    todo = {r["ID"]: [j for j in r[name] if (r["ID"], j) not in graded]
            for _, r in judge_df.iterrows()}
    print(f"[{name}] new pairs to judge: {sum(len(v) for v in todo.values())}")

    new = []
    for _, r in tqdm(judge_df.iterrows(), total=len(judge_df)):
        jobs = todo[r["ID"]]
        if not jobs:
            continue
        jobs_text = "\n\n".join(f"[J{n + 1}] {job_snippet(j)}" for n, j in enumerate(jobs))
        prompt = JUDGE_PROMPT.format(profile=profile_to_text(r["profile"]), jobs=jobs_text)
        grades = parse_json(llm(prompt, model=JUDGE_MODEL, system=JUDGE_SYSTEM))
        new += [{"ID": r["ID"], "job_idx": j, "grade": to_grade(grades.get(f"J{n + 1}"))}
                for n, j in enumerate(jobs)]

    out = pd.concat([grades_df, pd.DataFrame(new)], ignore_index=True)
    out.to_pickle(os.path.join(OUT_DIR, "judge_grades.pkl"))
    judge_df.to_pickle(os.path.join(OUT_DIR, "judge_candidates.pkl"))
    return out

def score_systems(systems):
    g = grades_df.set_index(["ID", "job_idx"])["grade"]
    rows = []
    for _, r in judge_df.iterrows():
        for s in systems:
            gr = np.array([g[(r["ID"], j)] for j in r[s]])
            lab = np.array([r["Category"] in corpus_labels[j] for j in r[s]])
            rows.append({"system": s, "label P@10": lab.mean(),
                         "judge P@10 (>=1)": (gr >= 1).mean(),
                         "judge P@10 (=2)": (gr == 2).mean()})
    print(pd.DataFrame(rows).groupby("system").mean().round(3).to_string())

grades_df = judge_system("top_seniority", "seniority_top10.pkl")
score_systems(["top_whole", "top_seniority"])

# Direct test: grades of jobs removed vs jobs added by the seniority penalty
g = grades_df.set_index(["ID", "job_idx"])["grade"]
removed, added = [], []
for _, r in judge_df.iterrows():
    removed += [g[(r["ID"], j)] for j in set(r["top_whole"]) - set(r["top_seniority"])]
    added += [g[(r["ID"], j)] for j in set(r["top_seniority"]) - set(r["top_whole"])]
print(f"\nSwapped jobs: {len(added)}")
print(f"  Removed jobs - mean grade {np.mean(removed):.2f}, share graded 2: {np.mean(np.array(removed) == 2):.0%}")
print(f"  Added jobs   - mean grade {np.mean(added):.2f}, share graded 2: {np.mean(np.array(added) == 2):.0%}")

[top_seniority] new pairs to judge: 27
               label P@10  judge P@10 (>=1)  judge P@10 (=2)
system                                                      
top_seniority       0.757             0.833            0.452
top_whole           0.755             0.814            0.452

Swapped jobs: 40
  Removed jobs - mean grade 0.93, share graded 2: 28%
  Added jobs   - mean grade 1.12, share graded 2: 28%


In [ ]:
grades_df = judge_system("top_clean", "clean_top10.pkl")
score_systems(["top_whole", "top_seniority", "top_clean"])

g = grades_df.set_index(["ID", "job_idx"])["grade"]
removed, added = [], []
for _, r in judge_df.iterrows():
    removed += [g[(r["ID"], j)] for j in set(r["top_seniority"]) - set(r["top_clean"])]
    added += [g[(r["ID"], j)] for j in set(r["top_clean"]) - set(r["top_seniority"])]
print(f"\nSwapped jobs: {len(added)}")
print(f"  Removed - mean grade {np.mean(removed):.2f}, share graded 2: {np.mean(np.array(removed) == 2):.0%}")
print(f"  Added   - mean grade {np.mean(added):.2f}, share graded 2: {np.mean(np.array(added) == 2):.0%}")

[top_clean] new pairs to judge: 23
               label P@10  judge P@10 (>=1)  judge P@10 (=2)
system                                                      
top_clean           0.755             0.836            0.443
top_seniority       0.757             0.833            0.452
top_whole           0.755             0.814            0.452

Swapped jobs: 40
  Removed - mean grade 1.32, share graded 2: 50%
  Added   - mean grade 1.25, share graded 2: 40%


## LLM ranking (kept)

`gpt-oss-20b` reorders the top 20 retrieved jobs and keeps 10. A different model (`gpt-oss-120b`) judges, so the system doesn't grade its own picks. Strict judge P@10 rises from 0.452 to **0.500**: better on 15 resumes, worse on 4 (sign test p = 0.019).

In this run, 3 of the 42 ranking responses were malformed JSON and fell back to the embedding order; the score counts them as they are. The app retries a failed ranking call once.

In [ ]:
corpus_clean = pd.read_pickle(os.path.join(OUT_DIR, "jobs_corpus_clean.pkl"))
top20 = pd.read_pickle(os.path.join(OUT_DIR, "seniority_top20.pkl"))

RANK_MODEL = "openai/gpt-oss-20b"
RANK_SYSTEM = "You are a career advisor matching a candidate to job postings. Respond with JSON only, no other text."

def clean_snippet(j, max_words=100):
    row = corpus_clean.iloc[j]
    return f"{row['title']}: {' '.join(str(row['description_clean']).split()[:max_words])}"

RANK_PROMPT = """Candidate profile:
{profile}

Job postings:
{jobs}

Tasks:
1. Rank the 10 jobs that best fit this candidate, best first. Consider role, skills, domain, and seniority.
2. For the top 5, also give:
   "why": 1-2 sentences on why it fits, referring to specific requirements in the posting
   "missing": up to 3 skills or requirements from the posting that the candidate lacks (empty list if none)
Use only information in the profile and the postings.

Return JSON in exactly this format, a single list of 10 jobs, best first:
{{"ranking": [{{"id": "J3", "why": "...", "missing": ["..."]}}, {{"id": "J7", "why": "...", "missing": []}}, {{"id": "J1"}}]}}
Include "why" and "missing" only for the first 5 jobs."""

def llm_rank(profile, cands):
    jobs_text = "\n\n".join(f"[J{n + 1}] {clean_snippet(j)}" for n, j in enumerate(cands))
    id_to_job = {f"J{n + 1}": j for n, j in enumerate(cands)}
    raw = llm(RANK_PROMPT.format(profile=profile_to_text(profile), jobs=jobs_text),
              model=RANK_MODEL, system=RANK_SYSTEM)
    try:
        items = parse_json(raw).get("ranking", [])
    except json.JSONDecodeError:
        return cands[:10], {}, 0   # parse failed: fall back to embedding order

    ranked, details = [], {}
    for it in items:
        jid = it.get("id") if isinstance(it, dict) else it
        j = id_to_job.get(str(jid).strip())
        if j is None or j in ranked:
            continue
        ranked.append(j)
        if isinstance(it, dict) and "why" in it:
            details[j] = {"why": it.get("why"), "missing": it.get("missing", [])}
    n_valid = len(ranked)
    ranked += [j for j in cands if j not in ranked]   # fill any gaps from embedding order
    return ranked[:10], details, n_valid

# Example: 2 resumes (CHEF and ENGINEERING)
for _, r in judge_df.drop_duplicates("Category").iloc[[3, 8]].iterrows():
    cands = top20[r["ID"]]
    top10, details, n_valid = llm_rank(r["profile"], cands)
    print(f"\n{'=' * 80}\n[{r['Category']}] {r['profile']['title']} ({r['profile']['seniority']})"
          f"  - valid IDs returned: {n_valid}/10")
    for rank, j in enumerate(top10, 1):
        print(f"{rank:2d}. (was #{cands.index(j) + 1:2d}) {corpus_clean.iloc[j]['title'][:65]}")
        if j in details:
            print(f"      why: {details[j].get('why')}")
            print(f"      missing: {details[j].get('missing')}")

[CHEF] Premium Club Chef (executive)  - valid IDs returned: 10/10
 1. (was # 2) Executive Chef
      why: Matches executive chef role in a high-end private club with banquet experience; candidate’s 23 years in fine dining and banquet leadership align with required high-quality, innovative operations.
      missing: []
 2. (was #12) Executive Chef
      why: Executive chef position in a prestigious club demanding leadership, menu creation, and staff training—all core strengths of the candidate.
      missing: []
 3. (was #14) Executive Chef
      why: The role requires overseeing multiple kitchen areas and driving financial performance, matching the candidate’s cost control and inventory management expertise.
      missing: []
 4. (was # 3) Corporate Chef
      why: Corporate chef overseeing multiple restaurants needs menu development, quality consistency, and profitability—areas where the candidate excels.
      missing: []
 5. (was #10) Executive Chef
      why: Executive chef role fo

In [ ]:
llm_top10, llm_details, n_valid_list = {}, {}, []
for _, r in tqdm(judge_df.iterrows(), total=len(judge_df)):
    top10, details, n_valid = llm_rank(r["profile"], top20[r["ID"]])
    llm_top10[r["ID"]] = [int(j) for j in top10]
    llm_details[r["ID"]] = {int(j): d for j, d in details.items()}
    n_valid_list.append(n_valid)

n_valid_arr = np.array(n_valid_list)
deep = np.mean([sum(top20[i].index(j) >= 10 for j in llm_top10[i]) for i in llm_top10])
print(f"Parse fallbacks: {(n_valid_arr == 0).sum()} | avg valid IDs: {n_valid_arr.mean():.1f}/10")
print(f"Avg jobs per top-10 pulled up from ranks 11-20: {deep:.1f}")

pd.to_pickle(llm_top10, os.path.join(OUT_DIR, "llm_top10.pkl"))
pd.to_pickle(llm_details, os.path.join(OUT_DIR, "llm_details.pkl"))

grades_df = judge_system("top_llm", "llm_top10.pkl")
score_systems(["top_whole", "top_seniority", "top_llm"])

Parse fallbacks: 3 | avg valid IDs: 8.8/10
Avg jobs per top-10 pulled up from ranks 11-20: 2.4
[top_llm] new pairs to judge: 62
               label P@10  judge P@10 (>=1)  judge P@10 (=2)
system                                                      
top_llm             0.745             0.881            0.500
top_seniority       0.757             0.833            0.452
top_whole           0.755             0.814            0.452


In [ ]:
from scipy.stats import binomtest

g = grades_df.set_index(["ID", "job_idx"])["grade"]
strict = {s: np.array([np.mean([g[(r["ID"], j)] == 2 for j in r[s]]) for _, r in judge_df.iterrows()])
          for s in ["top_seniority", "top_llm"]}
diff = strict["top_llm"] - strict["top_seniority"]
better, worse = (diff > 0).sum(), (diff < 0).sum()
print(f"top_llm vs top_seniority: better on {better}, worse on {worse}, tied on {(diff == 0).sum()} resumes")
print(f"Sign test p-value: {binomtest(better, better + worse).pvalue:.3f}")

top_llm vs top_seniority: better on 15, worse on 4, tied on 23 resumes
Sign test p-value: 0.019


## Explanation grounding

Are the listed gaps real requirements from the posting? The ranking call's explanations were too generous (82% listed no gaps). A separate explanation call, given 250 words of each posting instead of 100, cut empty gap lists to 4%, with 97% of listed gaps found in the posting text. The model sometimes returns malformed JSON, so the final parser reads each job object separately.

In [ ]:
import re

STOP = {"experience", "with", "and", "the", "for", "knowledge", "skills", "skill", "years",
        "year", "ability", "proficiency", "strong", "working", "using", "related", "degree"}

def terms(text):
    return [t for t in re.findall(r"[a-z][a-z+#]{2,}", text.lower()) if t not in STOP]

profiles = judge_df.set_index("ID")["profile"]
n_expl = sum(len(d) for d in llm_details.values())
n_empty = sum(1 for d in llm_details.values() for v in d.values() if not v.get("missing"))

total, grounded, flagged = 0, 0, []
for rid, d in llm_details.items():
    for j, v in d.items():
        shown = set(terms(clean_snippet(j)))
        for m in v.get("missing") or []:
            total += 1
            if any(t in shown for t in terms(m)):
                grounded += 1
            else:
                flagged.append((rid, j, m))

print(f"Explanations: {n_expl} | with empty 'missing' list: {n_empty} ({n_empty / n_expl:.0%})")
print(f"Missing skills listed: {total} | found in the job text shown: {grounded} ({grounded / max(total, 1):.0%})")

print(f"\n=== Flagged 'missing' skills (not found in job text), up to 5")
for rid, j, m in flagged[:5]:
    print(f"\n- MISSING: {m}\n  JOB: {clean_snippet(j)[:400]}...")

print(f"\n=== Read-through: 3 full explanations")
for rid in list(llm_details)[::14][:3]:
    j, v = next(iter(llm_details[rid].items()))
    print(f"\n--- CANDIDATE: {profiles[rid]['title']} ({profiles[rid]['seniority']})")
    print(f"    JOB TEXT SHOWN: {clean_snippet(j)}")
    print(f"    WHY: {v.get('why')}")
    print(f"    MISSING: {v.get('missing')}")

Explanations: 260 | with empty 'missing' list: 182 (70%)
Missing skills listed: 117 | found in the job text shown: 102 (87%)

=== Flagged 'missing' skills (not found in job text), up to 5

- MISSING: sales experience
  JOB: Float Relationship Banker: Relationship Banker Float What We Can Offer You Career Growth -- promotional opportunitiesIncentive program based on performancePaid Time Off (PTO), Paid Holidays for Full Time/Part Time EmployeesHealth, Dental, Vision, 401k match and Life InsuranceEmployee Assistance ProgramTuition Assistance Program (Full Time)Financial Coaching and Benefit GuidanceFloating Cultural Ho...

- MISSING: digital product knowledge
  JOB: Float Relationship Banker: Relationship Banker Float What We Can Offer You Career Growth -- promotional opportunitiesIncentive program based on performancePaid Time Off (PTO), Paid Holidays for Full Time/Part Time EmployeesHealth, Dental, Vision, 401k match and Life InsuranceEmployee Assistance ProgramTuition Assistance Progr

In [ ]:
EXPLAIN_PROMPT = """Candidate profile:
{profile}

Job postings:
{jobs}

For each job, write:
"why": 1-2 sentences naming specific requirements from that posting that the candidate meets
"missing": up to 3 requirements stated in that posting that the candidate's profile does not show
Rules: only name requirements that appear in the posting text above. Use an empty "missing" list only if the candidate clearly meets every stated requirement.

Return JSON in exactly this format:
{{"jobs": [{{"id": "J1", "why": "...", "missing": ["..."]}}, {{"id": "J2", "why": "...", "missing": []}}]}}"""

def snip(j, max_words):
    row = corpus_clean.iloc[j]
    return f"{row['title']}: {' '.join(str(row['description_clean']).split()[:max_words])}"

def explain(profile, jobs, max_words=250):
    jobs_text = "\n\n".join(f"[J{n + 1}] {snip(j, max_words)}" for n, j in enumerate(jobs))
    raw = llm(EXPLAIN_PROMPT.format(profile=profile_to_text(profile), jobs=jobs_text),
              model=RANK_MODEL, system=RANK_SYSTEM)
    try:
        items = parse_json(raw).get("jobs", [])
    except json.JSONDecodeError:
        return {}
    id_to_job = {f"J{n + 1}": j for n, j in enumerate(jobs)}
    out = {}
    for it in items:
        j = id_to_job.get(str(it.get("id")).strip()) if isinstance(it, dict) else None
        if j is not None:
            out[j] = {"why": it.get("why"), "missing": it.get("missing") or []}
    return out

def terms(text):   # fixed: keeps words with digits like "db2"
    return [t for t in re.findall(r"[a-z0-9+#]{3,}", text.lower()) if t not in STOP]

def grounding_stats(details, max_words):
    n_expl = n_empty = total = grounded = 0
    for rid, d in details.items():
        for j, v in d.items():
            n_expl += 1
            miss = v.get("missing") or []
            n_empty += not miss
            shown = set(terms(snip(j, max_words)))
            for m in miss:
                total += 1
                grounded += any(t in shown for t in terms(m))
    return (f"{n_expl} explanations | empty 'missing': {n_empty / n_expl:.0%} | "
            f"missing skills: {total}, found in job text: {grounded / max(total, 1):.0%}")

sample_ids = judge_df.drop_duplicates("Category").head(10)["ID"].tolist()
old = {rid: llm_details[rid] for rid in sample_ids}
new = {rid: explain(profiles[rid], llm_top10[rid][:5]) for rid in tqdm(sample_ids)}

print("OLD (ranking call, 100 words):     ", grounding_stats(old, 100))
print("NEW (explain call, 250 words):     ", grounding_stats(new, 250))

rid = sample_ids[1]
print(f"\n=== Example: {profiles[rid]['title']} ({profiles[rid]['seniority']})")
for j, v in new[rid].items():
    print(f"\n- {corpus_clean.iloc[j]['title']}\n  WHY: {v['why']}\n  MISSING: {v['missing']}")

OLD (ranking call, 100 words):      65 explanations | empty 'missing': 82% | missing skills: 13, found in job text: 100%
NEW (explain call, 250 words):      42 explanations | empty 'missing': 5% | missing skills: 115, found in job text: 97%

=== Example: Associate Consultant (senior)

- Mainframe Developer
  WHY: Candidate has over 7 years of mainframe experience, banking domain exposure, COBOL, JCL, CICS, DB2, VSAM, and Z/OS skills aligning with the required mainframe banking expertise.
  MISSING: ['Lending Experience', 'FIS ALS Experience', 'SharePoint usage']

- Senior Lead Software Engineer - Mainframe
  WHY: Candidate meets the required 5+ years of experience, CICS, COBOL, JCL, VSAM, DB2, and overall mainframe proficiency needed for the role.
  MISSING: ['Web services', 'BMS Maps and CICS Main view', 'REXX']

- Mainframe System Administrator
  WHY: Candidate has strong mainframe background with Z/OS, CICS, and DB2, satisfying core system administration capabilities.
  MISSING: ['M

In [ ]:
def parse_job_items(text):
    """Parse {"jobs": [...]}, tolerating several jobs merged into one object with repeated keys."""
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    top = dict(json.loads(text, object_pairs_hook=lambda pairs: pairs))
    items = []
    for obj in top.get("jobs", []):
        cur = {}
        for k, v in obj:          # pairs in order, repeats kept
            if k == "id" and "id" in cur:
                items.append(cur)
                cur = {}
            cur[k] = v
        if cur:
            items.append(cur)
    return items

def explain(profile, jobs, max_words=250):
    jobs_text = "\n\n".join(f"[J{n + 1}] {snip(j, max_words)}" for n, j in enumerate(jobs))
    raw = llm(EXPLAIN_PROMPT.format(profile=profile_to_text(profile), jobs=jobs_text),
              model=RANK_MODEL, system=RANK_SYSTEM)
    try:
        items = parse_job_items(raw)
    except (json.JSONDecodeError, TypeError, ValueError):
        return {}
    id_to_job = {f"J{n + 1}": j for n, j in enumerate(jobs)}
    out = {}
    for it in items:
        j = id_to_job.get(str(it.get("id")).strip())
        if j is not None:
            out[j] = {"why": it.get("why"), "missing": it.get("missing") or []}
    return out

new = {rid: explain(profiles[rid], llm_top10[rid][:5]) for rid in sample_ids}
for rid in sample_ids:
    print(f"{profiles[rid]['title'][:40]:<40} explanations: {len(new[rid])}/5")
print("\nNEW (explain call, 250 words):", grounding_stats(new, 250))

Payroll Accountant                       explanations: 5/5
Associate Consultant                     explanations: 5/5
Sales and Business Development           explanations: 5/5
Premium Club Chef                        explanations: 5/5
Project Construction Manager             explanations: 5/5
Consultant                               explanations: 5/5
Manager Assistant                        explanations: 5/5
Director of Digital Innovation and Strat explanations: 5/5
Mechanical Design Engineer               explanations: 5/5
Manager, Finance                         explanations: 5/5

NEW (explain call, 250 words): 50 explanations | empty 'missing': 4% | missing skills: 141, found in job text: 97%


## Export data for the app

In [5]:
import os, gzip, json
import numpy as np
import pandas as pd

BASE_DIR = r"C:\Users\LENOVO\Documents\Datasets"
OUT_DIR = os.path.join(BASE_DIR, "processed")
APP_DATA = os.path.join("app", "data")

corpus_clean = pd.read_pickle(os.path.join(OUT_DIR, "jobs_corpus_clean.pkl"))
emb = np.load(os.path.join(OUT_DIR, "jobs_emb_bge_small.npy"))
assert len(emb) == len(corpus_clean), "row count mismatch between jobs and embeddings"

def opt(x):
    return None if pd.isna(x) else str(x)

# Plain list of dicts (no pandas needed in the app); the app never reads past word 250 of a description
jobs_app = [{"title": str(r.title), "company": opt(r.company_name), "location": opt(r.location),
             "level": opt(r.formatted_experience_level),
             "desc": " ".join(str(r.description_clean).split()[:250])}
            for r in corpus_clean.itertuples()]

with gzip.open(os.path.join(APP_DATA, "jobs.json.gz"), "wt", encoding="utf-8") as f:
    json.dump(jobs_app, f, ensure_ascii=False)
np.save(os.path.join(APP_DATA, "jobs_emb.npy"), emb.astype(np.float32))

for f in sorted(os.listdir(APP_DATA)):
    print(f"{f}: {os.path.getsize(os.path.join(APP_DATA, f)) / 1e6:.1f} MB")

jobs.json.gz: 9.7 MB
jobs.parquet: 29.3 MB
jobs_emb.npy: 26.3 MB
samples.json: 0.0 MB


## Demo samples for the app

Four dataset resumes run through the full app pipeline (`app/pipeline.py`), saved so the demo buttons work without spending tokens.

In [ ]:
import sys
sys.path.insert(0, "app")
import pipeline

resumes_all = pd.read_pickle(os.path.join(OUT_DIR, "resumes_all.pkl"))

is_chef = resumes_all["text"].str.split().str[:3].str.join(" ").str.contains("CHEF")
chef_row = resumes_all[(resumes_all["Category"] == "CHEF") & is_chef].iloc[0]
print("Chef resume starts:", chef_row["text"][:80], "\n")

sample_rows = [resumes_all[resumes_all["Category"] == "ACCOUNTANT"].iloc[0],
               resumes_all[resumes_all["Category"] == "INFORMATION-TECHNOLOGY"].iloc[5],
               chef_row,
               resumes_all[resumes_all["Category"] == "ENGINEERING"].iloc[3]]

samples = []
for row in sample_rows:
    result = pipeline.match(row["text"].encode("utf-8"), "sample.txt")
    samples.append({"label": result["profile"]["title"], "result": result})
    n_expl = sum(1 for m in result["matches"] if m["why"])
    print(f"{row['Category']:24s} -> {result['profile']['title']} | top match: {result['matches'][0]['title']} | explained: {n_expl}/5")

with open(os.path.join("app", "data", "samples.json"), "w", encoding="utf-8") as f:
    json.dump(samples, f, ensure_ascii=False, indent=1)
print("\nSaved app/data/samples.json")

Chef resume starts: CHEF Career Focus I am a nursing student who has recently obtained my CNA licens 

ACCOUNTANT               -> Accountant | top match: Senior Accountant | explained: 5/5
INFORMATION-TECHNOLOGY   -> Information Technology Specialist | top match: Senior Help Desk | explained: 5/5
CHEF                     -> Private Chef | top match: Full-Time Resource Chef | explained: 5/5
ENGINEERING              -> Engineering Intern | top match: Mechanical Engineer Intern - Continuation | explained: 5/5

Saved app/data/samples.json


## ONNX Runtime check

The free host has 512 MB of RAM, too little for PyTorch, so the app runs bge-small on ONNX Runtime (`fastembed`). Same model and weights: embeddings match (cosine ≈ 1.000) and top-20 retrieval overlaps 19.9/20 on the evaluation set.

In [ ]:
import sys, time
import numpy as np
from fastembed import TextEmbedding
sys.path.insert(0, "app")
import pipeline

onnx_model = TextEmbedding("BAAI/bge-small-en-v1.5")
def onnx_encode(texts):
    e = np.array(list(onnx_model.embed(texts)), dtype=np.float32)
    return e / np.linalg.norm(e, axis=1, keepdims=True)

# Reference: the PyTorch model, set up exactly as it was used for all evaluations
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
st_model.max_seq_length = 512
st_model.half()

def st_encode(texts):
    return st_model.encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

# 1. Job embeddings: ONNX vs the stored PyTorch embeddings, same texts
corpus = pd.read_pickle(os.path.join(OUT_DIR, "jobs_corpus.pkl"))
idx = np.random.default_rng(0).choice(len(corpus), 200, replace=False)
texts = (corpus["title"].fillna("") + ". " + corpus["description"].fillna("")).iloc[idx].tolist()
cos = (onnx_encode(texts) * pipeline.job_emb[idx]).sum(axis=1)
print(f"Job embeddings, ONNX vs PyTorch cosine: mean {cos.mean():.4f}, min {cos.min():.4f}")

# 2. Retrieval: top-20 overlap on the 210 eval resumes (same query logic as pipeline.retrieve)
def top20(encode, text, prof):
    q = encode([text, pipeline.profile_to_text(prof)])
    combo = q[0] + q[1]
    combo /= np.linalg.norm(combo)
    s = pipeline.job_emb @ combo
    if prof["seniority"] in pipeline._job_bad:
        s = s - pipeline.LAMBDA * pipeline._job_bad[prof["seniority"]]
    return list(np.argsort(-s)[:20])

eval_clean = pd.read_pickle(os.path.join(OUT_DIR, "resumes_eval_clean_profiles.pkl"))
overlap, same_order, t_onnx = [], 0, 0.0
for _, r in eval_clean.iterrows():
    a = top20(st_encode, r["text"], r["profile"])
    t = time.time()
    b = top20(onnx_encode, r["text"], r["profile"])
    t_onnx += time.time() - t
    overlap.append(len(set(a) & set(b)))
    same_order += a == b

overlap = np.array(overlap)
print(f"Top-20 overlap: mean {overlap.mean():.2f}/20, min {overlap.min()}/20, "
      f"identical lists (same order): {same_order}/{len(overlap)}")
print(f"ONNX query time on your CPU: {t_onnx / len(overlap) * 1000:.0f} ms per resume")

Job embeddings, ONNX vs PyTorch cosine: mean 1.0002, min 0.9998
Top-20 overlap: mean 19.92/20, min 14/20, identical lists (same order): 120/210
ONNX query time on your CPU: 858 ms per resume
